# Summary_Day1.ipynb  
## 딥러닝 필수 파이썬 · 실습 환경구성 · OOP

이번 1강은 딥러닝을 바로 만드는 수업이라기보다,  
앞으로 PyTorch 코드를 읽기 위해 필요한 **파이썬 기초 체력**을 정리하는 강의이다.

강의 들으면서 기억해야 할 큰 흐름은 이것이다.

```text
딥러닝 학습 흐름
= Forward → Loss → Backward → Update
```

오늘 배운 내용은 다음 흐름으로 보면 된다.

1. 딥러닝 학습 구조 감 잡기
2. Loss, Gradient, Optimizer가 어디에 들어가는지 이해하기
3. Python 기본 자료형과 연산 확인하기
4. list, tuple, dict 같은 컨테이너 타입 이해하기
5. NumPy 배열과 copy()의 참조 문제 이해하기
6. 함수와 합성 함수로 딥러닝 구조 바라보기
7. 수치 미분과 PyTorch Autograd 연결하기
8. Class, 상속, `__call__` 이해하기
9. PyTorch 모델 구조 `nn.Module`로 이어서 생각하기

> 필기 포인트:  
> 1강은 코드를 많이 외우는 강의라기보다,  
> 앞으로 나오는 PyTorch 코드가 왜 그렇게 생겼는지 이해하는 준비 단계이다.

## 1. 기본 라이브러리 준비

이번 노트북에서는 NumPy, Matplotlib, PyTorch를 사용한다.

### 함수/모듈 사용법

```python
import numpy as np
import matplotlib.pyplot as plt
import torch
```

- `np`: NumPy를 짧게 쓰기 위한 별명이다.
- `plt`: 그래프를 그릴 때 쓰는 Matplotlib의 pyplot이다.
- `torch`: PyTorch 텐서와 자동 미분을 사용할 때 필요하다.

> 헷갈림 포인트:  
> `np`, `plt`는 함수 이름이 아니라 우리가 붙인 별명이다.  
> 그래서 `import numpy as np`라고 했기 때문에 `np.array()`처럼 쓸 수 있는 것이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

## 2. 딥러닝 학습 흐름 먼저 잡기

강의에서 사람의 학습과 컴퓨터의 학습을 비교했다.

사람은 보통 이렇게 공부한다.

```text
문제 풀기 → 채점하기 → 틀린 이유 확인 → 다시 풀기
```

딥러닝도 거의 비슷한다.

```text
Forward → Loss → Backward → Update
```

- `Forward`: 모델이 예측한다.
- `Loss`: 예측과 정답을 비교한다.
- `Backward`: 무엇을 얼마나 고쳐야 하는지 계산한다.
- `Update`: 파라미터를 수정한다.

> 시험 포인트:  
> 이 순서는 거의 모든 PyTorch 학습 코드에서 반복된다.

In [ ]:
learning_steps = [
    "1. Forward: 입력 데이터를 보고 예측한다",
    "2. Loss: 예측값과 정답을 비교해서 얼마나 틀렸는지 계산한다",
    "3. Backward: 틀린 이유를 거꾸로 추적해서 gradient를 계산한다",
    "4. Update: gradient를 이용해서 parameter를 수정한다",
]

for step in learning_steps:
    print(step)

## 3. Loss 개념

Loss는 모델이 얼마나 틀렸는지를 숫자로 표현한 값이다.

예를 들어:

```text
정답 = 5
예측 = 4
손실 = 1
```

Loss가 작아질수록 모델이 정답에 가까워지고 있다고 볼 수 있다.

### 직접 계산해보기

여기서는 아주 단순하게 절댓값 차이로 Loss를 계산해본다.

In [ ]:
target = 5
prediction = 4

loss = abs(target - prediction)

print("정답:", target)
print("예측:", prediction)
print("손실:", loss)

### 함수 사용법: `abs()`

```python
abs(x)
```

- `x`: 숫자
- 반환값: x의 절댓값

여기서 `abs(target - prediction)`을 쓴 이유는  
예측이 정답보다 크든 작든, 일단 얼마나 떨어져 있는지를 보고 싶기 때문이다.

> 참고:  
> 실제 딥러닝에서는 `MSELoss`, `CrossEntropyLoss` 같은 손실 함수를 쓴다.

## 4. Optimizer와 Parameter Update

파라미터 업데이트 공식은 다음처럼 생각하면 된다.

```text
Param = Param - (lr * grad)
```

- `Param`: 모델이 학습하는 값이다. 예: weight, bias
- `lr`: learning rate, 학습률이다.
- `grad`: gradient, 기울기이다.

> 내 식으로 이해하기:  
> gradient가 “이쪽으로 가면 Loss가 커져요”라고 알려주면,  
> 우리는 그 반대 방향으로 조금 움직이다.

In [ ]:
param = 10.0
lr = 0.1
grad = 2.0

new_param = param - (lr * grad)

print("수정 전 param:", param)
print("learning rate:", lr)
print("gradient:", grad)
print("수정 후 param:", new_param)

### 변수 이름 정리

| 변수 | 뜻 | 기억법 |
|---|---|---|
| `param` | parameter | 모델이 학습하는 값 |
| `lr` | learning rate | 한 번에 움직이는 크기 |
| `grad` | gradient | 수정 방향을 알려주는 기울기 |

> 주의:  
> `lr`이 너무 크면 최적점을 지나쳐서 발산할 수 있고,  
> 너무 작으면 학습이 너무 느립니다.

## 5. Python 기본 자료형

파이썬에서 자주 나오는 기본 자료형이다.

| 자료형 | 예시 | 의미 |
|---|---|---|
| int | `1` | 정수 |
| float | `2.03` | 실수 |
| str | `'abc'` | 문자열 |
| bool | `True` | 참/거짓 |

### 함수 사용법: `type()`

```python
type(value)
```

- `value`: 자료형을 확인할 값
- 반환값: 그 값의 타입

In [ ]:
a = 1
b = 2.03
c = "abc"
d = True

print(a, type(a))
print(b, type(b))
print(c, type(c))
print(d, type(d))

> 헷갈림 포인트:  
> `print(a)`는 값을 보여주는 것이고,  
> `type(a)`는 a가 어떤 종류의 데이터인지 보여준다.

## 6. 기본 연산자

파이썬에서 자주 쓰는 연산이다.

| 연산 | 코드 | 의미 |
|---|---|---|
| 더하기 | `+` | 덧셈 |
| 빼기 | `-` | 뺄셈 |
| 곱하기 | `*` | 곱셈 |
| 나누기 | `/` | 실수 나눗셈 |
| 몫 | `//` | 나눈 몫 |
| 나머지 | `%` | 나눈 나머지 |
| 거듭제곱 | `**` | 제곱 |

In [ ]:
print("1 + 2 =", 1 + 2)
print("3 - 2 =", 3 - 2)
print("3 * 5 =", 3 * 5)
print("6 / 2 =", 6 / 2)
print("13 // 5 =", 13 // 5)
print("13 % 5 =", 13 % 5)
print("2 ** 3 =", 2 ** 3)

> 시험보다 실습에서 더 중요한 포인트:  
> 딥러닝 코드에서는 `** 2`가 자주 나온다.  
> 보통 오차를 제곱할 때 쓴다.

예:

```python
loss = ((pred - y) ** 2).mean()
```

## 7. List와 NumPy Array 차이

파이썬 list와 NumPy array는 비슷해 보이지만 연산 방식이 다릅니다.

### list

```python
[1, 2, 3] + [4, 5, 6]
```

리스트끼리 붙다.

### NumPy array

```python
np.array([1, 2, 3]) + np.array([4, 5, 6])
```

같은 위치의 숫자끼리 더한다.

> 딥러닝에서는 숫자 배열 연산이 중요하므로 NumPy / Tensor 방식에 익숙해져야 한다.

In [ ]:
python_list_result = [1, 2, 3] + [4, 5, 6]

numpy_array_result = np.array([1, 2, 3]) + np.array([4, 5, 6])

print("Python list 결과:", python_list_result)
print("NumPy array 결과:", numpy_array_result)

### 함수 사용법: `np.array()`

```python
np.array([1, 2, 3])
```

- 괄호 안에는 리스트나 튜플 같은 데이터를 넣는다.
- 반환값은 NumPy 배열이다.

> 기억할 점:  
> list는 일반 데이터 묶음이고,  
> NumPy array는 수치 계산에 최적화된 배열이다.

## 8. List 인덱싱과 슬라이싱

리스트에서 특정 위치의 값을 꺼내는 것을 인덱싱이라고 한다.

```python
list1[0]
```

구간을 꺼내는 것을 슬라이싱이라고 한다.

```python
list1[1:4]
```

- 앞 숫자: 시작 위치
- 뒤 숫자: 끝 위치, 단 끝 위치는 포함하지 않음

In [ ]:
list1 = [1, 2, 3, 4, 5]

print("전체:", list1)
print("첫 번째 값:", list1[0])
print("마지막 값:", list1[-1])
print("1번부터 3번 전까지:", list1[1:3])
print("전체 복사처럼 보기:", list1[:])
print("거꾸로 보기:", list1[::-1])

> 헷갈림 포인트:  
> `list1[::-1]`은 원본을 바꾸지 않고 거꾸로 보여주는 방식이다.  
> 반면 `list1.reverse()`는 원본 자체를 바꾼다.

In [ ]:
list2 = [1, 2, 3, 4, 5]

print("슬라이싱으로 거꾸로:", list2[::-1])
print("슬라이싱 후 원본:", list2)

list2.reverse()

print("reverse() 후 원본:", list2)

### 함수 사용법: `reverse()`

```python
list1.reverse()
```

- 리스트 원본을 직접 뒤집다.
- 반환값을 새로 주는 함수가 아니라, 원본을 수정한다.

> 주의:  
> 원본 데이터를 보존해야 하는 상황에서는 `reverse()`처럼 원본을 바꾸는 함수를 조심해야 한다.

## 9. Tuple

Tuple은 list와 비슷하지만 값을 바꿀 수 없다.

```python
t = (1, 2, 3)
```

> 내 느낌으로 정리하면:  
> list는 수정 가능한 묶음, tuple은 고정된 묶음이다.

In [ ]:
t = (1, 2, 3, 4, 5)

print(t)
print(type(t))
print("길이:", len(t))
print("1번 인덱스:", t[1])

### 함수 사용법: `len()`

```python
len(container)
```

- 리스트, 튜플, 문자열 등의 길이를 반환한다.

> 헷갈림 포인트:  
> `t[1] = 10`처럼 튜플 값을 바꾸려고 하면 에러가 납니다.  
> 튜플은 한 번 만들면 내부 값을 직접 수정할 수 없다.

## 10. Dictionary

Dictionary는 key와 value를 한 쌍으로 저장한다.

```python
my_dict = {"yes": 1, "no": 0}
```

딥러닝이나 데이터 처리에서 label mapping 할 때 자주 본다.

예:

```python
{"cat": 0, "dog": 1}
```

In [ ]:
my_dict = {"yes": 1, "no": 0}

print(my_dict)
print("yes 값:", my_dict["yes"])

my_dict["neutral"] = 3

print("추가 후:", my_dict)

### 함수 사용법: `.items()`

```python
for key, value in my_dict.items():
    ...
```

- 딕셔너리의 key와 value를 함께 꺼냅니다.
- 반복문에서 자주 쓴다.

In [ ]:
kor_eng = {"사랑": "love", "두산": "doosan"}

for key, value in kor_eng.items():
    print(key, ":", value)

> 기억할 점:  
> dictionary는 JSON 구조와도 비슷해서,  
> API 데이터나 설정 파일을 읽을 때 자주 만나게 된다.

## 11. for문과 if문

반복과 조건은 거의 모든 코드의 기본이다.

### for문 사용법

```python
for item in container:
    실행할 코드
```

### if문 사용법

```python
if 조건:
    조건이 참일 때 실행
else:
    조건이 거짓일 때 실행
```

In [ ]:
for i in range(1, 6):
    if i % 2 == 0:
        print(i, "짝수입니다")
    else:
        print(i, "홀수입니다")

### 함수 사용법: `range()`

```python
range(start, end)
```

- `start`: 시작 숫자
- `end`: 끝 숫자, 단 포함하지 않음

예:

```python
range(1, 6)
```

이면 1, 2, 3, 4, 5가 나온다.

## 12. 함수 만들기

함수는 반복해서 쓸 코드를 이름 붙여 저장하는 방식이다.

```python
def 함수이름(인자):
    실행할 코드
    return 결과
```

> 딥러닝 관점에서 보면, 모델도 결국 입력을 받아 출력을 내는 함수처럼 볼 수 있다.

In [ ]:
def square(x):
    p2 = x * x
    return p2

x1 = 12
result = square(x1)

print("입력:", x1)
print("제곱 결과:", result)

### 함수 사용법: `def`

```python
def square(x):
    return x * x
```

- `square`: 함수 이름
- `x`: 함수에 넣는 값, 즉 인자
- `return`: 계산 결과를 밖으로 돌려줌

> 헷갈림 포인트:  
> `print()`는 화면에 보여주는 것이고,  
> `return`은 함수 결과를 실제로 밖으로 넘기는 것이다.

## 13. 함수에서 여러 값 반환하기

파이썬 함수는 여러 값을 한 번에 반환할 수 있다.

실제로는 tuple 형태로 반환된다고 생각하면 된다.

In [ ]:
def square_and_cube(x):
    p2 = x * x
    p3 = x * x * x
    return p2, p3

x1 = 12
x2, x3 = square_and_cube(x1)

print("입력:", x1)
print("제곱:", x2)
print("세제곱:", x3)

> 기억할 점:  
> 나중에 모델 평가 함수에서 `loss, accuracy`처럼 여러 값을 반환하는 경우가 많다.

## 14. 컨테이너 타입의 함정: 얕은 복사

NumPy 배열은 데이터를 담는 그릇이기도 하지만,  
변수는 실제 값 전체를 들고 있다기보다 **메모리 위치를 가리키는 참조**라고 볼 수 있다.

그래서 단순히:

```python
y = x
```

라고 하면 새 복사본이 생기는 게 아니라,  
`x`와 `y`가 같은 데이터를 같이 바라보게 된다.

In [ ]:
x = np.array([5, 7, 9])

y = x

print("처음 x:", x)
print("처음 y:", y)

x[1] = -1

print("x 수정 후 x:", x)
print("x 수정 후 y:", y)

결과를 보면 `x`만 수정했는데 `y`도 같이 바뀐다.

> 실무적으로 위험한 이유:  
> 데이터 전처리를 하다가 원본 데이터가 같이 바뀌면,  
> 실험 결과가 꼬일 수 있다.

## 15. copy()로 안전하게 복사하기

원본 데이터를 보존해야 한다면 `copy()`를 사용한다.

### 함수 사용법: `.copy()`

```python
y = x.copy()
```

- `x`와 같은 값을 가진 새 배열을 만듭니다.
- 원본과 복사본이 따로 관리된다.

In [ ]:
x = np.array([5, 7, 9])

y = x.copy()

x[1] = -1

print("x:", x)
print("y:", y)

이번에는 `x`를 바꿔도 `y`는 그대로이다.

> 시험 포인트:  
> `y = x`는 참조 복사,  
> `y = x.copy()`는 안전한 복사이다.

## 16. Tensor와 NumPy 변환 시 주의점

PyTorch Tensor를 NumPy 배열로 바꿀 수 있다.

### 함수 사용법

```python
tensor.numpy()
```

또는 예전 코드에서는:

```python
tensor.data.numpy()
```

를 볼 수 있다.

그런데 이때도 메모리를 공유할 수 있으니 조심해야 한다.

In [ ]:
x1 = torch.ones(5)

x2 = x1.numpy()

print("처음 x1:", x1)
print("처음 x2:", x2)

x1[1] = -1

print("수정 후 x1:", x1)
print("수정 후 x2:", x2)

Tensor에서 NumPy로 바꿨는데도 값이 같이 바뀌었다.

> 기억할 점:  
> Tensor와 NumPy 변환은 편하지만,  
> 원본을 안전하게 보존하려면 `.copy()`까지 쓰는 습관이 좋다.

In [ ]:
x1 = torch.ones(5)

x2 = x1.numpy().copy()

x1[1] = -1

print("x1:", x1)
print("x2:", x2)

## 17. 함수 그래프 그리기: 2차 함수

이제 함수와 그래프를 연결해서 본다.

이번 함수:

```text
f(x) = 2x² + 2
```

이런 식으로 함수를 정의하고, 여러 x값을 넣어서 y값을 계산한다.

In [ ]:
def f(x):
    return 2 * x**2 + 2

x = np.arange(-2, 2.1, 0.25)
y = f(x)

print("x:")
print(x)

print("\ny:")
print(y)

### 함수 사용법: `np.arange()`

```python
np.arange(start, stop, step)
```

- `start`: 시작값
- `stop`: 끝값, 단 stop은 보통 포함되지 않음
- `step`: 간격

여기서는 -2부터 2까지 0.25 간격으로 x값을 만들었다.

In [ ]:
plt.plot(x, y, label="f(x) = 2x^2 + 2")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Quadratic Function")
plt.legend()
plt.show()

그래프 해석:

- U자 모양이다.
- x가 0일 때 가장 낮다.
- x가 양쪽으로 멀어질수록 y가 커집니다.

> 딥러닝 연결:  
> 나중에 Loss 그래프도 이런 식으로 최저점을 찾아 내려가는 문제로 생각할 수 있다.

## 18. 합성 함수

딥러닝 모델은 거대한 합성 함수이다.

```text
y = f3(f2(f1(x)))
```

입력이 여러 함수를 차례대로 지나면서 출력이 만들어진다.

강의에서는 이걸 “함수들의 블록 조립”처럼 설명했다.

In [ ]:
def f1(x):
    return x**2

def f2(x):
    return 2 * x

def f3(x):
    return x + 2

x_value = 3

y_value = f3(f2(f1(x_value)))

print("결과:", y_value)

계산 흐름:

```text
x = 3
f1(3) = 9
f2(9) = 18
f3(18) = 20
```

> 기억할 점:  
> PyTorch 모델에서 `model(x)`도 결국 이런 함수 연결이라고 보면 된다.

In [ ]:
x = np.arange(-2, 2.1, 0.25)

x1 = f1(x)
x2 = f2(x1)
y = f3(x2)

plt.plot(x, y, label="f3(f2(f1(x)))")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Composite Function")
plt.legend()
plt.show()

## 19. 미분은 왜 필요한가?

미분은 “어느 방향으로 고쳐야 하는가?”를 알려준다.

딥러닝에서는 Loss를 줄이고 싶다.

그래서 필요한 정보는:

```text
파라미터를 키워야 하나?
줄여야 하나?
얼마나 바꿔야 하나?
```

이걸 알려주는 신호가 Gradient이다.

## 20. 수치 미분: 중앙 차분

중앙 차분 공식:

```text
f'(x) ≈ (f(x+h) - f(x-h)) / 2h
```

x를 기준으로 양쪽의 아주 가까운 값을 보고 기울기를 근사한다.

### 함수 사용법 관점

우리는 `fdiff(f)`라는 함수를 만들 겁니다.

```python
diff = fdiff(f)
diff(3.0)
```

- `fdiff(f)`: 함수 f를 받아서 미분 함수 diff를 돌려줌
- `diff(3.0)`: x=3에서의 기울기 계산

In [ ]:
def fdiff(f):
    def diff(x):
        h = 1e-6
        return (f(x + h) - f(x - h)) / (2 * h)

    return diff

diff = fdiff(f)

x_point = 3.0
slope = diff(x_point)

print("x =", x_point)
print("수치 미분값:", slope)

결과는 거의 12가 나온다.

왜냐하면:

```text
f(x) = 2x² + 2
f'(x) = 4x
f'(3) = 12
```

> 주의:  
> 이 코드는 미분 원리를 이해하려고 만든 것이다.  
> 실제 딥러닝에서는 PyTorch Autograd가 자동으로 해준다.

## 21. 함수와 도함수 그래프 비교

원래 함수와 미분 결과를 같이 그려본다.

In [ ]:
x = np.arange(-2, 2.1, 0.25)
y = f(x)

diff = fdiff(f)
y_dash = diff(x)

plt.plot(x, y, label="f(x)")
plt.plot(x, y_dash, label="f'(x)")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Function and Derivative")
plt.legend()
plt.show()

그래프 해석:

- `f(x)`는 U자 모양이다.
- `f'(x)`는 직선이다.
- x가 음수이면 기울기도 음수이다.
- x가 양수이면 기울기도 양수이다.
- x=0 근처에서 기울기는 0이다.

> 딥러닝 연결:  
> 기울기가 0인 지점은 더 이상 내려갈 방향이 거의 없는 지점, 즉 최저점 후보이다.

## 22. Sigmoid 함수와 미분

Sigmoid는 나중에 이진 분류에서 다시 나온다.

공식:

```text
g(x) = 1 / (1 + exp(-x))
```

출력값이 항상 0과 1 사이이다.

In [ ]:
def g(x):
    return 1 / (1 + np.exp(-x))

x = np.arange(-6, 6.1, 0.25)
y = g(x)

plt.plot(x, y, label="sigmoid")
plt.xlabel("x")
plt.ylabel("g(x)")
plt.title("Sigmoid Function")
plt.legend()
plt.show()

### 함수 사용법: `np.exp()`

```python
np.exp(x)
```

- 자연상수 e의 x제곱을 계산한다.
- Sigmoid 공식에서 자주 나온다.

> 기억할 점:  
> Sigmoid는 큰 음수는 0에 가깝게, 큰 양수는 1에 가깝게 바꾼다.

In [ ]:
diff_g = fdiff(g)
y_dash = diff_g(x)

plt.plot(x, y, label="sigmoid")
plt.plot(x, y_dash, label="sigmoid derivative")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Sigmoid and Derivative")
plt.legend()
plt.show()

그래프 해석:

- Sigmoid는 가운데에서 가장 많이 변한다.
- 양 끝으로 갈수록 기울기가 작아집니다.
- 이 특성은 나중에 기울기 소실 문제와도 연결된다.

## 23. PyTorch Autograd 미리보기

PyTorch에서는 직접 수치 미분을 구현하지 않아도 된다.

`requires_grad=True`를 설정하면 PyTorch가 계산 과정을 추적한다.

### 함수 사용법

```python
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x**2 + 2
y.backward()
x.grad
```

- `requires_grad=True`: 이 Tensor를 미분 대상으로 추적
- `backward()`: 역전파 실행
- `.grad`: 계산된 미분값 확인

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = 2 * x**2 + 2

y.backward()

print("y:", y)
print("x.grad:", x.grad)

결과는 `tensor(12.)`이다.

앞에서 수치 미분으로 구한 값과 같다.

> 기억할 점:  
> 우리가 손으로 만든 `fdiff()`는 원리 이해용,  
> 실제 PyTorch 학습에서는 `loss.backward()`가 자동으로 gradient를 계산한다.

## 24. Gradient 누적 주의

PyTorch의 gradient는 기본적으로 누적된다.

그래서 학습 루프에서는 매번:

```python
optimizer.zero_grad()
```

또는 직접 Tensor에서는:

```python
x.grad.zero_()
```

를 해줘야 한다.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = 2 * x**2 + 2
y.backward()
print("첫 번째 grad:", x.grad)

y = 2 * x**2 + 2
y.backward()
print("두 번째 backward 후 grad:", x.grad)

x.grad.zero_()
print("zero_() 후 grad:", x.grad)

### 함수 사용법: `zero_()`

```python
x.grad.zero_()
```

- gradient 값을 0으로 직접 바꾼다.
- `_`가 붙은 PyTorch 함수는 보통 원본을 직접 수정한다.

> 헷갈림 포인트:  
> gradient가 자동으로 새로 덮어써지는 게 아니라 누적된다.  
> 그래서 학습 루프에서 초기화가 꼭 필요하다.

## 25. Class 기본 개념

PyTorch 모델을 이해하려면 Class를 알아야 한다.

| 개념 | 의미 |
|---|---|
| Class | 객체를 만들기 위한 설계도 |
| Instance | 클래스로 만든 실제 객체 |
| Attribute | 객체가 가진 데이터 |
| Method | 객체가 가진 함수 |

> 쉽게 말하면:  
> Class는 붕어빵 틀, Instance는 실제 붕어빵이다.

## 26. Point 클래스 만들기

2차원 좌표의 점을 나타내는 클래스를 만들어본다.

### 문법

```python
class 클래스이름:
    def __init__(self, ...):
        초기 설정

    def 메서드이름(self, ...):
        동작
```

- `__init__`: 객체가 만들어질 때 처음 실행된다.
- `self`: 자기 자신, 즉 만들어진 객체를 의미한다.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def move(self, dx, dy):
        self.x += dx
        self.y += dy

    def coord(self):
        return (self.x, self.y)

p = Point(2, 3)

print("처음 좌표:", p.coord())

p.move(1, -1)

print("이동 후 좌표:", p.coord())

코드 흐름:

```text
p = Point(2, 3)
→ p.x = 2, p.y = 3

p.move(1, -1)
→ x는 1 증가, y는 1 감소

결과: (3, 2)
```

> 기억할 점:  
> 클래스는 데이터와 기능을 한 묶음으로 관리하려고 쓴다.

## 27. Class로 그림 그리기

이번에는 Point 클래스에 `draw()` 메서드를 추가해서 그래프에 점을 찍어본다.

### 함수 사용법: `plt.plot()`

```python
plt.plot(x, y, marker='o')
```

- `x`: x좌표
- `y`: y좌표
- `marker`: 점 모양

In [ ]:
class DrawablePoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def draw(self):
        plt.plot(self.x, self.y, marker="o", markersize=10)

p1 = DrawablePoint(2, 3)
p2 = DrawablePoint(-1, -2)

p1.draw()
p2.draw()

plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.title("Point Class Example")
plt.show()

> PyTorch 연결:  
> 나중에 `class MyModel(nn.Module)`도 이런 식으로  
> 데이터 흐름과 기능을 클래스 안에 묶어서 정의한다.

## 28. 상속: Circle 클래스 만들기

상속은 기존 클래스의 기능을 물려받는 방식이다.

```python
class ChildClass(ParentClass):
    ...
```

여기서는 Point의 x, y 좌표 기능을 물려받고,  
Circle에서는 반지름 r을 추가한다.

In [ ]:
import matplotlib.patches as patches

class Circle(DrawablePoint):
    def __init__(self, x, y, r):
        super().__init__(x, y)
        self.r = r

    def draw(self):
        super().draw()
        circle = patches.Circle(
            xy=(self.x, self.y),
            radius=self.r,
            fill=False
        )
        ax = plt.gca()
        ax.add_patch(circle)

c = Circle(1, 0, 2)

c.draw()

plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.gca().set_aspect("equal")
plt.title("Circle inherits Point")
plt.show()

### 함수 사용법: `super()`

```python
super().__init__(x, y)
```

- 부모 클래스의 `__init__`을 실행한다.
- 이미 만들어진 기능을 다시 쓰고 싶을 때 사용한다.

> 기억할 점:  
> PyTorch 모델에서도 `super().__init__()`이 거의 항상 나온다.  
> `nn.Module`의 기본 기능을 제대로 쓰기 위해 필요하다.

## 29. `__call__` 이해하기

클래스 안에 `__call__`을 만들면 객체를 함수처럼 부를 수 있다.

```python
h = H()
h(3)
```

이렇게 쓰면 내부적으로:

```python
h.__call__(3)
```

이 실행된다.

In [ ]:
class H:
    def __call__(self, x):
        return 2 * x**2 + 2

h = H()

result = h(3)

print("h(3) 결과:", result)

> PyTorch 핵심 연결:  
> 우리가 모델을 사용할 때 보통 `model.forward(x)`라고 직접 쓰지 않고  
> `model(x)`라고 쓴다.  
> 이게 가능한 이유가 `__call__` 구조 때문이다.

In [ ]:
x = np.arange(-2, 2.1, 0.25)

h = H()
y = h(x)

plt.plot(x, y)
plt.xlabel("x")
plt.ylabel("h(x)")
plt.title("Callable Class H")
plt.show()

## 30. PyTorch 모델 구조 예고

PyTorch 모델은 보통 이런 구조이다.

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = ...

    def forward(self, x):
        return ...
```

- `__init__`: 필요한 Layer를 준비한다.
- `forward`: 데이터가 Layer를 통과하는 순서를 정의한다.
- 사용할 때는 `model(x)`처럼 호출한다.

In [ ]:
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(1, 1)

    def forward(self, x):
        return self.layer(x)

model = SimpleModel()

sample_x = torch.tensor([[1.0], [2.0], [3.0]])

sample_output = model(sample_x)

print(model)
print("입력 shape:", sample_x.shape)
print("출력 shape:", sample_output.shape)
print(sample_output)

### 함수 사용법: `nn.Linear()`

```python
nn.Linear(in_features, out_features)
```

- `in_features`: 입력 feature 개수
- `out_features`: 출력 feature 개수

여기서는 입력값 1개를 받아 출력값 1개를 만듭니다.

> 앞으로 나올 거의 모든 모델은 이 구조에서 시작한다고 보면 된다.

## 31. PyTorch 학습 루프 모양만 보기

아직 실제 학습은 다음 차시부터 더 자세히 하겠지만,  
1강에서는 학습 루프의 모양만 잡아두면 된다.

In [ ]:
import torch.optim as optim

model = SimpleModel()

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

x_train = torch.tensor([[1.0], [2.0], [3.0]])
y_train = torch.tensor([[2.0], [4.0], [6.0]])

optimizer.zero_grad()

pred = model(x_train)

loss = criterion(pred, y_train)

loss.backward()

optimizer.step()

print("prediction:")
print(pred)

print("loss:", loss.item())

### 함수 사용법 정리

```python
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)
```

- `nn.MSELoss()`: 예측값과 정답의 평균 제곱 오차를 계산한다.
- `optim.SGD(...)`: gradient를 이용해 파라미터를 수정한다.
- `model.parameters()`: 모델 안의 weight와 bias를 Optimizer에게 넘깁니다.
- `lr`: learning rate이다.

학습 루프:

```text
zero_grad → pred → loss → backward → step
```

> 시험 포인트:  
> PyTorch 학습 순서를 외우기보다,  
> “이전 기울기 지우고 → 예측하고 → 채점하고 → 역추적하고 → 고친다”로 이해하면 된다.

## 32. 환경 구성 관련 메모

원본 실습에는 Colab에서 폰트 설치, torchviz 설치, tree 설치, 이미지 다운로드 같은 명령어가 있었다.

예:

```python
!sudo apt-get install -y fonts-nanum*
!pip install torchviz
!git clone ...
```

이런 코드는 주로 Colab/Linux 환경에서 실행하는 명령이다.

> 필기 느낌으로 정리하면:  
> 모델 공부의 핵심은 아니고,  
> 그래프 한글 폰트 설정이나 외부 자료 다운로드를 위해 필요한 준비 코드이다.

이번 Summary에서는 모든 환경에서 최대한 안전하게 실행되도록  
외부 다운로드가 필요한 코드는 제외하고 핵심 Python/PyTorch 구조만 정리했다.

## 33. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `np` | NumPy 별명 | `np.array()`, `np.arange()` |
| `plt` | Matplotlib pyplot | `plt.plot()`, `plt.show()` |
| `torch` | PyTorch | `torch.tensor()`, `torch.ones()` |
| `nn` | neural network 모듈 | `nn.Linear()`, `nn.MSELoss()` |
| `optim` | optimizer 모듈 | `optim.SGD()` |
| `x` | 입력값 | 함수나 모델에 넣는 값 |
| `y` | 정답/출력값 | 예측해야 하는 값 |
| `pred` | prediction | 모델의 예측값 |
| `loss` | 손실 | 예측이 얼마나 틀렸는지 |
| `grad` | gradient | 수정 방향 |
| `lr` | learning rate | 한 번에 움직이는 크기 |
| `epoch` | 반복 학습 단위 | 전체 데이터를 한 번 학습 |
| `forward` | 순전파 | 입력에서 예측으로 가는 과정 |
| `backward` | 역전파 | Loss에서 거꾸로 gradient 계산 |
| `copy()` | 복사 | 원본 보호용 복사 |
| `__init__` | 생성자 | 객체 만들 때 초기 설정 |
| `__call__` | 호출 메서드 | 객체를 함수처럼 호출 |
| `self` | 자기 자신 | 객체 내부 속성/메서드 접근 |

## 34. 시험용 요약

```text
딥러닝 학습 = Forward → Loss → Backward → Update
```

꼭 기억할 것:

- Loss는 모델이 얼마나 틀렸는지를 나타냅니다.
- Gradient는 어느 방향으로 고쳐야 하는지를 알려준다.
- Optimizer는 gradient를 이용해 parameter를 수정한다.
- 학습률 `lr`이 너무 크면 발산하고, 너무 작으면 느립니다.
- list와 NumPy array는 연산 방식이 다릅니다.
- `y = x`는 같은 데이터를 참조할 수 있다.
- 원본을 보호하려면 `copy()`를 사용한다.
- 딥러닝 모델은 거대한 합성 함수이다.
- 수치 미분은 원리 이해용이고, 실제로는 PyTorch Autograd를 쓴다.
- `requires_grad=True`는 자동 미분 추적을 시작한다.
- `backward()`는 gradient를 계산한다.
- PyTorch gradient는 누적되므로 `zero_grad()` 또는 `zero_()`가 필요하다.
- Class는 데이터와 기능을 묶는 설계도이다.
- `__call__` 덕분에 `model(x)`처럼 모델을 함수처럼 사용할 수 있다.
- PyTorch 모델은 보통 `nn.Module`을 상속해서 만듭니다.
- `__init__`에서는 Layer를 준비하고, `forward()`에서는 데이터 흐름을 정의한다.